# Bending StyleGAN activations

In this notebook, we will bend the activations of the model, instead of the weights. As we said in the first notebook, activations are intermediary values that are computed during the generation process. The big advantage of activation bending it can be dependent on the inputs, and can then be made adaptive to the input. While most of the bending operations implemented so far are still a little "brutal", we will see some examples where this aspect can be interesting : interpolation and adaptive masking. 

In [3]:
%load_ext autoreload
%autoreload 1

import re, urllib, os, sys
import torch
from pathlib import Path


import panel as pn
pn.extension()


torchbend_dir = Path(os.getcwd()).parent / ".." 
os.environ['TORCHBEND_DEFAULT_MODEL_DIR'] = str(torchbend_dir / "models")
models_dir = torchbend_dir / "models" / "stylegan3" 
os.makedirs(models_dir, exist_ok=True)
sys.path.append(str(torchbend_dir.resolve()))
torch.set_grad_enabled(False)

%aimport torchbend
tb = torchbend

from torchbend.interfaces.stylegan import BendedStyleGAN
tb.set_output('notebook')

# download the model from website
model_url = "https://api.ngc.nvidia.com/v2/models/org/nvidia/team/research/stylegan3/1/files?redirect=true&path=stylegan3-r-ffhqu-256x256.pkl"
# set device
# device = torch.device('cpu')
device =  torch.device('cuda:0')
bended = BendedStyleGAN(model_url, device=device)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Simple activation bending

We will apply the same bending operations as in the second notebook, but this time on activations. The effects are a little bit similar than weight bending, as the performed bending operations are almost mathematically equivalent in this precise case : ie, for a convolution operation, scaling the kernel or the input is similar as convolution is linear. However, this can be very different for different neural architectures, and you will still notice some differences in the provided artifacts. For exemple, here the `Noise` callback does not corrput the kernel, but the incoming tensors, and then create spatial artifacts (the kernel is the same for all the image, and then cannot do so).

In [4]:

bended.reset()

target_layer = 11
bend_affine_layers = False

inputs = bended.get_inputs()

scripted = bended.script()
scripted(**inputs)


c1 = tb.BendingParameter("mask", 1., range=[0., 1.])
c2 = tb.BendingParameter("seed", 0, range=[0, 1024])
cb_mask_kernels = tb.OrderedMask(prob = c1, seed = c2, dim=1)

c3 = tb.BendingParameter("scale", 1., range=[-5, 5])
c4 = tb.BendingParameter("bias", 0., range=[-5, 5])
cb_affine_kernels = tb.Affine(scale=c3, bias=c4)

c5 = tb.BendingParameter("noise", 0., range=[0., 5.])
cb_noise_kernels = tb.Normal(std=c5)

c6 = tb.BendingParameter("permute_seed", -1, range=[-1, 1024])
cb_permute_kernels = tb.Permute(seed=c6, dim=1)

target_activation = bended.aliases()['layer_out'][target_layer]

bended.bend(cb_mask_kernels, target_activation, bend_param=False)
bended.bend(cb_affine_kernels, target_activation, bend_param=False)
bended.bend(cb_noise_kernels, target_activation, bend_param=False)
bended.bend(cb_permute_kernels, target_activation, bend_param=False)


print(bended.model.bended_activations())


panel = tb.ui.panel.panel_generation_ui(bended, realtime=True, input=(tuple(), inputs), script=True, norm_fn=lambda x: (x.clamp(-1, 1) + 1) / 2)
panel


{'filtered_lrelu_10': [OrderedMask(prob=1.000), Affine(scale=1.0000, bias=0.0000), Normal(std=0.000, seed=0), Permute(dim=1)]}


BokehModel(combine_events=True, render_bundle={'docs_json': {'aa415176-d074-4bea-a9fc-6e7d3a9393ac': {'version…

## Adaptive activation bending

This adaptive aspect of activation bending is really important, as it allows to drive the bending operation with the actual values of the target. For example, we can use the `tb.Lambda` callback to provide an arbitrary transformation on the input, that we can make dependent of the value. 

Here, we provide two examples of adaptive transformations of an input : 

- *threshold_reverse* : flips spatially the channels with channels with lower amplitude (and hence less likely to contribute to the final image).
- *remove_uncorrelated* : computes the correlation between each channels of the input, and removes `param`% of the less correlated channels.
- *random_recompose* : randomly recomposes patches from the input tensor

By playing with the parameter, you'll get a touch of how we benefit from activation bending, by being able to recompose space, or to design bendings that are based on the input statistics. 

In [8]:
%autoreload 1
from functools import partial
from itertools import product
from torchbend.ui.panel import panel_generation_ui
from torchbend.utils import channel_correlation, upper_threshold, prime_factors
import math

bended.reset()

target_layer = 10
target_activation = bended.aliases()['layer_out'][target_layer]

def threshold_reverse(x, threshold: torch.Tensor | None = None):
    # activations are batches x channels x width x height
    # here, we will reverse the channels whose maximum values are lower 
    #   than threshold * global_max
    if threshold is None:
        threshold = torch.tensor(0.).to(x)
    threshold = threshold.abs() * x.abs().amax()
    x_channel_max = x.abs().amax((2, 3))
    for i in range(x.size(0)):
        for j in range(x.size(1)):
            if x_channel_max[i, j] < threshold:
                x[i, j] = x[i, j].fliplr().flipud()
    return x


def mask_from_channel_correlation(x, correlation: torch.Tensor | None = None, mode: str = "lower"):
    if correlation is None:
        correlation =  torch.tensor(0.).to(x)
    correlation = correlation.to(x)
    corr_score = channel_correlation(x)
    if mode == "upper":
        corr_mask = corr_score > upper_threshold(corr_score.flatten().float(), 1 - correlation.item())
    else:
        corr_mask = corr_score < upper_threshold(corr_score.flatten().float(), correlation.item())
    x = corr_mask[..., None, None] * x    
    return x


# fetch a possible number of patches from the activation shape
act_props = list(bended.activations(target_activation).values())[0]
decomposition = list(filter(lambda x: x < 7, tb.prime_factors(act_props.shape[-1])))
n_patches = math.prod(decomposition) if len(decomposition) <= 2 else math.prod(decomposition[::2])

def random_recompose(x, seed: float = 0, n_patches: int | None = None):
    seed = int(seed)
    if seed == -1:
        return x
    assert x.shape[-1] % n_patches == 0
    assert n_patches is not None
    dim_patches = int(x.shape[-1] / n_patches)
    interp_weights = torch.bernoulli(torch.full((n_patches**2, x.size(0), x.size(0)), 0.5)).to(x)
    x_interp = x.reshape(x.size(0), n_patches ** 2, -1).permute(1, 0, 2)
    x_out = torch.bmm(interp_weights, x_interp).permute(1, 0, 2).reshape(x.shape)
    return x_out

random_recompose = partial(random_recompose, n_patches=n_patches)


# c1 = tb.BendingParameter("correlation_threshold", 0., range=[0., 1.])
# cb_reverse = tb.Lambda(mask_from_channel_correlation, param=c1)
# bended.bend(cb_reverse, target_activation)

c2 = tb.BendingParameter("reverse_threshold", 0., range=[0, 1])
cb_correlation = tb.Lambda(threshold_reverse, param=c2)
bended.bend(cb_correlation, target_activation)

# c3 = tb.BendingParameter("random_recompose", -1., range=[-1, 1024])
# cb_random_recompose = tb.Lambda(random_recompose, param=c3)
# bended.bend(cb_random_recompose, target_activation)

# scripted = bended.script()
panel_generation_ui(bended, realtime=False, script=False, input=(tuple(), bended.get_inputs()))



BokehModel(combine_events=True, render_bundle={'docs_json': {'f983f91b-4257-45fa-80ef-750afcfdbd21': {'version…

TypeError: BendingParameter values can only be int or float

TypeError: BendingParameter values can only be int or float